# NB13 — Visualizaciones de Presentación
**ZMM Movilidad Predictiva**

Consolida visualizaciones finales: heatmap M1, comparación M2 vs M3, dashboard hotspots, pipeline.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

RUTA_PROCESSED = '../data_processed/'
RUTA_OUTPUTS   = '../outputs/'

print('='*65)
print('NB13: VISUALIZACIONES DE PRESENTACION')
print('='*65)

In [ ]:
# 1. Heatmap Clusters de Horas (M1)
df = pd.read_csv(RUTA_PROCESSED + 'super_tabla_con_clusters.csv', parse_dates=['fecha_hora'])
df = df[(df['fecha_hora'] >= '2023-01-01') & (df['fecha_hora'] <= '2025-12-31 23:00:00')].copy()

feats_m1 = ['hora_del_dia','dia_semana','es_fin_de_semana','intensidad_hora_pico',
            'temperatura_c','nivel_lluvia','impacto_evento_activo','asistencia_estimada',
            'siniestros_zona_industrial']
perfil = df.groupby('cluster_hora')[feats_m1].mean()
perfil_z = (perfil - perfil.mean()) / perfil.std()
perfil_z.columns = [c.replace('_',' ').title() for c in perfil_z.columns]

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(perfil_z.T, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax,
            cbar_kws={'label': 'Desviaciones del promedio'})
ax.set_title('Perfil de los 5 Clusters Horarios (M1)', fontsize=14)
plt.tight_layout(); plt.savefig(RUTA_OUTPUTS + 'viz_heatmap_clusters_horas.png', dpi=150); plt.show()

In [ ]:
# 2. Comparacion Feature Importance M2 vs M3
with open(RUTA_OUTPUTS + 'modelo_m2_rf.pkl', 'rb') as f: m2 = pickle.load(f)
with open(RUTA_OUTPUTS + 'modelo_m3_rf.pkl', 'rb') as f: m3 = pickle.load(f)

fi2 = m2['feature_importance'].rename(columns={'importance':'M2 Clasificacion'}).set_index('feature')
fi3 = m3['feature_importance'].rename(columns={'importance':'M3 Regresion'}).set_index('feature')
fi = fi2.join(fi3, how='outer').fillna(0)
fi = fi.loc[fi.sum(axis=1).sort_values(ascending=False).head(12).index]

fig, ax = plt.subplots(figsize=(12, 7))
x = np.arange(len(fi)); w = 0.35
ax.barh(x-w/2, fi['M2 Clasificacion'], w, label='M2 Clasificacion', color='steelblue')
ax.barh(x+w/2, fi['M3 Regresion'], w, label='M3 Regresion', color='coral')
ax.set_yticks(x); ax.set_yticklabels(fi.index); ax.invert_yaxis()
ax.set_xlabel('Importancia'); ax.set_title('Feature Importance: M2 vs M3'); ax.legend()
plt.tight_layout(); plt.savefig(RUTA_OUTPUTS + 'viz_comparacion_modelos.png', dpi=150); plt.show()

In [ ]:
# 3. Dashboard Hotspots
hs = pd.read_csv(RUTA_OUTPUTS + 'hotspots_top15.csv')
fig, axes = plt.subplots(1, 3, figsize=(18, 8))

lbl = [f'{i+1}.{c[:30]}' for i,c in enumerate(hs['combinacion'])]
axes[0].barh(range(len(hs)), hs['riesgo_promedio'], color='coral')
axes[0].set_yticks(range(len(hs))); axes[0].set_yticklabels(lbl, fontsize=8)
axes[0].set_xlabel('Riesgo Promedio'); axes[0].set_title('Top 15 Hotspots'); axes[0].invert_yaxis()

axes[1].barh(range(len(hs)), hs['siniestros_reales'], color='steelblue')
axes[1].set_yticks(range(len(hs))); axes[1].set_yticklabels(lbl, fontsize=8)
axes[1].set_xlabel('Siniestros Reales'); axes[1].set_title('Siniestros Historicos'); axes[1].invert_yaxis()

axes[2].scatter(hs['riesgo_promedio'], hs['siniestros_reales'], s=hs['total_horas']/5,
    c=range(len(hs)), cmap='RdYlGn_r', alpha=0.8, edgecolors='black')
for _, r in hs.iterrows():
    axes[2].annotate(f'#{r["rank"]}', (r['riesgo_promedio'], r['siniestros_reales']), fontsize=7, ha='center', va='bottom')
axes[2].set_xlabel('Riesgo Predicho'); axes[2].set_ylabel('Siniestros Reales')
axes[2].set_title('Riesgo vs Realidad'); axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.savefig(RUTA_OUTPUTS + 'viz_dashboard_hotspots.png', dpi=150); plt.show()

In [ ]:
# 4. Pipeline Diagram
fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(0,10); ax.set_ylim(0,4); ax.set_aspect('equal'); ax.axis('off')

bloques = [
    (0.5,2.0,'DATOS\n(rativ + clima\n+ empresas)','#e0e0e0'),
    (2.3,2.0,'NB01-06\nFeat. Eng.\nsuper_tabla','#b3d4fc'),
    (4.1,3.0,'NB07: M1\nK-Means\nHoras','#90EE90'),
    (4.1,1.0,'NB08: M2\nK-Means\nZonas','#90EE90'),
    (5.9,2.0,'NB09: M2\nXGBoost\nClasif.','#FFD700'),
    (7.7,3.0,'NB11\nSHAP','#FFA07A'),
    (7.7,1.0,'NB12\nHotspots','#FFA07A'),
    (9.5,2.0,'NB13\nVisualizacion\nFinal','#DDA0DD'),
]
for x,y,txt,col in bloques:
    ax.add_patch(plt.Rectangle((x-0.6,y-0.5),1.2,1.0,facecolor=col,edgecolor='black',lw=1.5,zorder=2))
    ax.text(x,y,txt,ha='center',va='center',fontsize=7,fontweight='bold',zorder=3)

for x1,y1,x2,y2 in [(1.1,2,1.7,2),(3.5,2.3,3.5,2.8),(3.5,1.7,3.5,1.2),
                     (4.7,2,5.3,2),(6.5,2.3,7.1,2.8),(6.5,1.7,7.1,1.2),(8.3,2,8.9,2)]:
    ax.annotate('',xy=(x2,y2),xytext=(x1,y1),arrowprops=dict(arrowstyle='->',color='black',lw=1.5))

ax.set_title('Pipeline Completo: ZMM Movilidad Predictiva', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout(); plt.savefig(RUTA_OUTPUTS + 'viz_pipeline_completo.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# 5. README
readme = """VISUALIZACIONES GENERADAS \u2014 ZMM Movilidad Predictiva\n=====================================================\n\nNB08 Clustering Zonas:\n  - nb08_seleccion_k_zonas.png\n  - mapa_zonas_clusters.html\n\nNB09 Clasificacion M2:\n  - m2_matriz_confusion.png\n  - m2_feature_importance.png\n\nNB10 Regresion M3:\n  - m3_real_vs_predicho.png\n  - m3_feature_importance.png\n\nNB11 SHAP:\n  - shap_summary_plot.png\n  - shap_dependence_hora.png\n  - shap_dependence_pico.png\n  - shap_waterfall_alto_riesgo.png\n\nNB12 Hotspots:\n  - hotspots_visualizacion.png\n  - hotspots_top15.csv\n  - hotspots_presentacion.csv\n\nNB13 Consolidacion:\n  - viz_heatmap_clusters_horas.png\n  - viz_comparacion_modelos.png\n  - viz_dashboard_hotspots.png\n  - viz_pipeline_completo.png"""

with open(RUTA_OUTPUTS + 'README_visualizaciones.txt', 'w', encoding='utf-8') as f:
    f.write(readme)
print('Saved: README_visualizaciones.txt')
print('\nNB13 COMPLETADO')